In [2]:
%pip install sqlalchemy mysql-connector-python

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 47.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
   ---------------------------------------- 0/2 [sqlalchemy]
  

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Load & Inspect Raw Data

In [4]:
# Load dataset with standard e-commerce encoding
df_raw = pd.read_csv('data.csv', encoding='ISO-8859-1')

print(f"Total Raw Rows: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")

# View the first 5 rows
df_raw.head()

Total Raw Rows: 541,909
Total Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Checking Missing Values & Data Types

In [5]:
# Check non-null counts and data types
df_raw.info()

print("\n--- Missing Values Count ---")
print(df_raw.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB

--- Missing Values Count ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Data Cleaning

In [6]:
df_clean = df_raw.copy()

# 1. Drop rows with missing CustomerID (Retention & RFM requires identified customers)
df_clean = df_clean.dropna(subset=['CustomerID'])
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# 2. Filter out negative or zero Quantities (Cancellations and returns start with 'C' in InvoiceNo)
df_clean = df_clean[df_clean['Quantity'] > 0]

# 3. Filter out zero or negative UnitPrice
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# 4. Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Rows before cleaning: {len(df_raw):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Rows removed:         {len(df_raw) - len(df_clean):,}")

Rows before cleaning: 541,909
Rows after cleaning:  397,884
Rows removed:         144,025


## Feature Engineering

In [7]:
# 1. Calculate Total Amount per line item
df_clean['TotalAmount'] = (df_clean['Quantity'] * df_clean['UnitPrice']).round(2)

# 2. Extract year-month for Cohort grouping
df_clean['InvoiceMonth'] = df_clean['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

# 3. Clean string columns (strip whitespace, title casing)
df_clean['Description'] = df_clean['Description'].astype(str).str.strip().str.title()
df_clean['Country'] = df_clean['Country'].astype(str).str.strip()

# View the engineered columns
df_clean[['InvoiceNo', 'CustomerID', 'InvoiceDate', 'Quantity', 'UnitPrice', 'TotalAmount', 'InvoiceMonth']].head()

,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalAmount,InvoiceMonth
0,536365,17850,2010-12-01 08:26:00,6,2.55,15.30,2010-12-01
1,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,2010-12-01
2,536365,17850,2010-12-01 08:26:00,8,2.75,22.00,2010-12-01
3,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,2010-12-01
4,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,2010-12-01


## Push Data into MySQL

In [8]:
# Configure your MySQL connection details
DB_USER = "root"                    # Your MySQL username (default is root)
DB_PASSWORD = "1234" # <-- REPLACE WITH YOUR REAL MYSQL PASSWORD
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ecommerce_db"

# Create SQLAlchemy connection engine
connection_url = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

print("🚀 Pushing cleaned data to MySQL... (takes ~30-45 seconds)")

# Push DataFrame into MySQL table 'ecom_transactions'
table_name = "ecom_transactions"
df_clean.to_sql(
    name=table_name,
    con=engine,
    if_exists='replace',   # Overwrites table if it already exists
    index=False,
    chunksize=10000        # Loads in batches of 10,000 rows
)

print(f"✅ Successfully loaded {len(df_clean):,} records into MySQL table `{table_name}`!")

🚀 Pushing cleaned data to MySQL... (takes ~30-45 seconds)
✅ Successfully loaded 397,884 records into MySQL table `ecom_transactions`!


## Verify the Data in MySQL

In [9]:
# Query MySQL to verify the records
query = """
SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT CustomerID) AS total_customers,
    ROUND(SUM(TotalAmount), 2) AS total_revenue
FROM ecom_transactions;
"""

df_summary = pd.read_sql(query, con=engine)
df_summary

,total_records,total_customers,total_revenue
0,397884,4338,8911407.9
